# exp009 公榜推理 Notebook

端到端赛道提交（公榜：仅加载权重推理，不重训）。

**随本 Notebook 一起上传的文件（缺一无效）**

| 文件 | 作用 |
|---|---|
| `exp009_predict.py` | 特征重建 + 采样 + `main(datasources, start_date, end_date)` + 输出自检 |
| `exp009_model.py` | 模型结构与 JSON 权重存取（训练侧定义原样副本） |
| `exp009_model.json` | 文本权重（结构超参 + train 期 mean/std + state_dict，~49MB） |

**模型**：exp009 BaselineTransformer（27ch × 16 根 30m bar 窗口，~2.25M 参数，训练 2020-01~2023-12，seed=42；验证集 IC 0.079 / ICIR 0.590（真实标签））。

**口径要点**
- 27 通道 = 25 个基础特征（复权对数收益 10 + 成交量 log1p 差分 15，仅原始字段的基础数据工程）+ bar_valid + gap_flag；标准化统计量取自权重文件（train 期统计，无泄漏）。
- 窗口 = 截面时刻对应的 30m bar（结束时刻=截面时刻）向前共 16 根，窗口完全早于标签窗口；有效 bar < 8 的窗口按训练口径丢弃。
- 覆盖全部 8 个截面（9:30~14:30）：9:30 按方案 B1 近似，以竞价 bar（结束于 09:25）为窗口末位、末位允许无效、slot_id 借用 10:00 嵌入，数据严格早于截面时刻无泄漏；排序才重要，平台统一做去极值/标准化/风格剔除。

In [ ]:
# 平台推理入口：main(datasources, start_date, end_date)
from exp009_predict import main

平台将以 `main(datasources, start_date, end_date)` 调用推理：
- `datasources`：逻辑名 → 评测表名映射，本模型只使用 `bar30m`；
- `start_date / end_date`：评测区间（按自然日，双端包含）。

返回三列 `date, instrument, factor`（已按契约清理：±inf→NaN→dropna→去重，并跑过自检清单）。

如需在平台手动自评（可选）：

```python
factor_data = main(datasources, start_date, end_date)
M.jyc_eval._latest(factor_data=factor_data)
```